# Dephased-IC dataset: parallel generation

Generates recordings on the `single-knob-dephased-ic` branch. Each recording
warm-starts from a state snapshot instead of `h.finitialize(-65)`, which removes
the shared initial condition that ignites the flagship network once at ~4.9 s.

**Run the cells in order.** Cell 2 is a preflight that will tell you if anything
is wrong before you spend compute. Cell 3 is the long one.

### The dephasing works

5-recording pilot, all four gating checks passed:

| check | measured |
|---|---|
| population Vm excursion at 4–6 s | **+0.74 mV** (flagship: +10.30 mV) |
| bursts in the flagship's 4.60–5.34 s band | **0 of 6** |
| mean rate | 0.2927 Hz vs flagship 0.2789 (**+4.9%**) |
| V_rest | −82.65 mV vs flagship −83.31 |

### One known artifact, and why `DISCARD_EXTRA_MS` is not optional

The warm start restores **membrane** state (`v`, hh gates, kA gates, `ko`, sAHP)
but **not synaptic** state. `h.finitialize()` zeroes every synaptic conductance
and resets the depression resource to `R = 1`, so each recording rebuilds
recurrent conductance from zero over ~1 s while running at maximum recurrent
gain. In roughly **1 in 3** recordings that ignites a ~57%-participation burst.

Measured properties:

- **Timing is pinned**: sim 960–1030 ms in every case, i.e. file-clock 0–30 ms
  under the flagship's 1000 ms discard — the first bin of kept data.
- **Not snapshot-locked**: 2 of 4 noise seeds burst on the *same* snapshot, so
  adding snapshots does **not** help.
- **Indistinguishable in shape** from a genuine spontaneous burst
  (see `analysis/dephase_marked_rasters_B_zoom.png`), so nothing downstream
  would flag it.

`DISCARD_EXTRA_MS = 3000` moves the kept window past both the burst and the
post-burst suppression trough. Costs ~5% wall clock and no kept data.

The alternative fix — snapshotting and restoring synaptic state too
(`g_ampa`, `g_nmda`, `R`, inhibitory and noise `g`) — removes the rebuild at
source and needs no discard, but requires a fresh ~2.7 h warm-up run. Worth doing
if anything downstream cares about the first seconds of each recording.

Evidence: `analysis/dephase_marked_rasters_C_diagnostic.png`.

## 1. Configuration

Four values you set; the rest have defaults that match the flagship.

### What the warm-start parameters mean

This branch exists because `h.finitialize(-65)` starts every one of the 926 cells
in the *same* artificial state — zero adaptation — which makes the whole population
ignite together at ~4.9 s in every recording. The fix is to start each recording
from a state the network actually reaches on its own:

| parameter | meaning |
|---|---|
| `WARMUP_MS = 130000` | run the network **once** for 130 s so it settles into its natural ongoing state |
| `SNAPSHOT_TIMES = [50, 70, 90, 110, 130] s` | save the full state of all 926 cells at these moments during that run |

Recordings then start from those saved states. Values are set against
`tau_slow = 6.5 s`: 50 s in is ~8 time constants (settled), and 20 s apart is
~3 time constants (so the five snapshots genuinely differ from each other).

Five snapshots for 50 recordings means each is reused 10 times — but always with a
different noise stream, so no two recordings are identical. Building more costs
~2.5 h each and, measured, would **not** remove the residual settling burst.

### The other defaults

- `NOISE_SEED_BASE = 1000` — matches the flagship, so dephased recording NNN sees
  the same noise as flagship recording NNN and the two differ *only* in the
  initial condition. Change it if you want independent noise.
- `DISCARD_EXTRA_MS = 3000` — not optional; see the artifact note at the top.
- `VOLTAGE = 'probe'` — 40 cells at 5 ms (~3 MB/recording). `'all'` is ~77 MB.
- Everything else — topology, weights, `noise_weight`, `tau_k`, `sahp_ainc_fast`,
  `sahp_tau_slow` — comes from the flagship worker config unchanged, so the two
  states are the same network differing in one parameter.

In [ ]:
# ===========================================================================
# SET THESE
# ===========================================================================
SAHP_NORMAL, SAHP_SEIZURE = 0.01, 0.004   # the single knob; the ONLY difference
STATES_TO_RUN = ['normal', 'seizure']     # trim to do one state at a time
N_RECORDINGS  = 50                        # per state
N_WORKERS     = 5                         # concurrent NEURON processes


# ===========================================================================
# LEAVE ALONE unless you have a reason  (see the markdown above for why)
# ===========================================================================
DURATION_MS      = 60000.0    # kept length per recording; matches the flagship
NOISE_SEED_BASE  = 1000       # matches the flagship, so recording NNN sees the
                              # same noise as flagship NNN (only the IC differs)

VOLTAGE          = 'probe'    # 'probe' ~3 MB/rec | 'all' ~77 MB | 'none'
VOLTAGE_PROBE_N  = 40
VOLTAGE_DT       = 5.0

WARMUP_MS        = 130000.0   # warm-start: one settling run per state
SNAPSHOT_TIMES   = [50000., 70000., 90000., 110000., 130000.]   # states saved here
DISCARD_EXTRA_MS = 3000.0     # drops the synaptic-rebuild burst at sim ~1 s

STATES = {'normal': SAHP_NORMAL, 'seizure': SAHP_SEIZURE}

print('states to run:', STATES_TO_RUN)
for s in STATES_TO_RUN:
    print('  %-8s sahp_ainc_slow = %.4f uS' % (s, STATES[s]))
print('')
print('%d recordings x %.0f s per state, %d workers, voltage=%s'
      % (N_RECORDINGS, DURATION_MS / 1000, N_WORKERS, VOLTAGE))
print('noise seed base %d   discard %.0f ms   warm-up %.0f s, %d snapshots'
      % (NOISE_SEED_BASE, 1000.0 + DISCARD_EXTRA_MS,
         WARMUP_MS / 1000, len(SNAPSHOT_TIMES)))

## 2. Preflight — run this before committing compute

In [ ]:
import os, sys, glob, json, subprocess, time
import numpy as np

REPO = os.path.abspath('..') if os.path.exists(os.path.join('..', 'neuron_simulation')) else os.path.abspath('.')
ANALYSIS = os.path.join(REPO, 'analysis')
DEPHASED = os.path.join(REPO, 'notebooks', 'NEURON data parallel', 'dephased_ic')
PY       = sys.executable

def lib_path(state):  return os.path.join(ANALYSIS, 'dephase_state_library_%s.npz' % state)
def out_dir(state):   return os.path.join(DEPHASED, state)

ok, need_library = True, []
print('repo    :', REPO)
print('python  :', PY, '(%s)' % sys.version.split()[0])
assert os.path.exists(os.path.join(REPO, 'neuron_simulation')), \
    'could not locate the repo root; open this notebook from notebooks/ or the repo root'

# --- the workers run THIS interpreter, so it must have NEURON --------------
probe = subprocess.run([PY, '-c', 'import neuron; print(neuron.__version__)'],
                       capture_output=True, text=True)
if probe.returncode == 0:
    print('neuron  :', probe.stdout.strip(), 'OK')
else:
    ok = False
    print('\nFAIL: this kernel cannot import neuron. Select the Python 3.9')
    print('      interpreter that has NEURON (Select Kernel -> Python Environments...).')

mech = subprocess.run([PY, '-c', 'import sys; sys.path.insert(0, r"%s");'
                       'from neuron_simulation.neurons import load_mechanisms;'
                       'load_mechanisms(); print("ok")' % REPO],
                      capture_output=True, text=True, cwd=REPO)
if mech.returncode == 0:
    print('mechs   : OK')
else:
    ok = False
    print('\nFAIL: mechanisms not loadable. Build:  cd neuron_simulation && nrnivmodl mechanisms')

# --- per-state status ------------------------------------------------------
print('\n%-9s %-10s %-34s %s' % ('state', 'knob', 'warm-start library', 'recordings'))
print('-' * 78)
todo = {}
for s in STATES_TO_RUN:
    lp = lib_path(s)
    if os.path.exists(lp):
        lb = np.load(lp)
        lk = float(lb['sahp_ainc_slow']) if 'sahp_ainc_slow' in lb else None
        n_snap = lb['g_slow'].shape[0]
        if lk is not None and abs(lk - STATES[s]) > 1e-12:
            libtxt = 'MISMATCH: built at %.4f' % lk
            ok = False
        else:
            libtxt = '%d snapshots, sd %.5f uS' % (n_snap, lb['g_slow'].std(axis=1).mean())
    else:
        libtxt = 'MISSING -> run the next cell'
        need_library.append(s)
    d = out_dir(s)
    have = len(glob.glob(os.path.join(d, 'recording*.npz')))
    todo[s] = [r for r in range(N_RECORDINGS)
               if not os.path.exists(os.path.join(d, 'recording%03d.npz' % r))]
    print('%-9s %-10.4f %-34s %d have / %d to go'
          % (s, STATES[s], libtxt, have, len(todo[s])))

print('\noutput folders:')
for s in STATES_TO_RUN:
    print('  %-9s %s' % (s, out_dir(s)))

sim_s = (DURATION_MS + 1000.0 + DISCARD_EXTRA_MS) / 1000.0
solo_min = sim_s * 68.4 / 60.0
n_todo = sum(len(v) for v in todo.values())
print('\nmeasured 68.4x realtime -> ~%.0f min per %.0f s recording, solo' % (solo_min, sim_s))
print('generation: %d recordings total -> ~%.1f h with %d workers'
      % (n_todo, n_todo * solo_min / 60.0 / max(1, N_WORKERS), N_WORKERS))
if need_library:
    lib_h = len(need_library) * (WARMUP_MS/1000.0) * 68.4 / 3600.0
    print('libraries : %d to build -> ~%.1f h (they can run in parallel)'
          % (len(need_library), lib_h))
mb = {'all': 77.0, 'probe': 3.0, 'none': 0.6}[VOLTAGE]
print('disk      : ~%.2f GB (%s voltage)' % (n_todo * mb / 1024.0, VOLTAGE))

print('\nPREFLIGHT %s' % ('OK' if ok else 'FAILED - fix the above first'))
if need_library:
    print('run the LIBRARY cell next for: %s' % ', '.join(need_library))

## 2b. Build the warm-start libraries (once per state, ~2.5 h each)

Skip this if preflight showed both libraries present. Each state gets its own
because the seizure network's stationary state differs from normal's — warm-starting
one from the other's snapshots would add a relaxation transient.

Both states can warm up concurrently, so two states cost about the same wall clock
as one.

In [ ]:
if not need_library:
    print('all requested libraries already present - skip this cell')
else:
    snaps = [str(int(t)) for t in SNAPSHOT_TIMES]
    procs = []
    for s in need_library:
        log = os.path.join(ANALYSIS, '_dephase_lib_%s.log' % s)
        cmd = [PY, '-u', os.path.join(ANALYSIS, 'dephase_snapshot.py'),
               '--state', s, '--sahp-ainc-slow', str(STATES[s]),
               '--noise-seed-base', str(NOISE_SEED_BASE),
               '--duration', str(WARMUP_MS), '--snapshots'] + snaps
        procs.append((s, subprocess.Popen(cmd, stdout=open(log, 'w'),
                                          stderr=subprocess.STDOUT), log))
        print('warming up %-8s (sahp_ainc_slow=%.4f) -> %s'
              % (s, STATES[s], os.path.basename(log)))
        time.sleep(15)

    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        print('[%5.1f min] warming up...' % ((time.time() - t0) / 60), flush=True)
        time.sleep(180)

    for s, p, log in procs:
        print('  %-8s exit=%s' % (s, p.returncode))
        if p.returncode:
            for line in open(log).read().strip().splitlines()[-6:]:
                print('     ' + line)

    print('')
    print('libraries now on disk:')
    for s in STATES_TO_RUN:
        lp = lib_path(s)
        if os.path.exists(lp):
            lb = np.load(lp)
            knob = float(lb['sahp_ainc_slow']) if 'sahp_ainc_slow' in lb else float('nan')
            print('  %-8s %d snapshots, knob %.4f, within-population sd %.5f uS'
                  % (s, lb['g_slow'].shape[0], knob, lb['g_slow'].std(axis=1).mean()))
        else:
            print('  %-8s STILL MISSING' % s)
    print('')
    print('re-run the preflight cell before generating.')

## 3. Generate (the long cell)

Launches `N_WORKERS` subprocesses, each taking a contiguous slice of recording
indices. Resumable — rerun the cell and it skips whatever already exists. Worker
logs go to `analysis/_dephase_nb_w*.log`; the cell polls and prints progress.

In [ ]:
assert ok, 'preflight failed - do not run this cell'
assert not need_library, 'build the warm-start libraries first (cell 2b)'

# One worker slice per (state, chunk). Workers are shared across states, so
# N_WORKERS processes run at a time regardless of how many states are queued.
jobs = []
for s in STATES_TO_RUN:
    if not todo[s]:
        print('%-8s: nothing to do' % s)
        continue
    per = int(np.ceil(len(todo[s]) / max(1, N_WORKERS)))
    for w in range(N_WORKERS):
        chunk = todo[s][w*per:(w+1)*per]
        if chunk:
            jobs.append((s, w, chunk[0], len(chunk)))
print('\n%d worker jobs:' % len(jobs))
for s, w, start, count in jobs:
    print('  %-8s w%d: recordings %d..%d' % (s, w, start, start + count - 1))

running, done_jobs, t0 = [], [], time.time()
queue = list(jobs)
while queue or running:
    while queue and len(running) < N_WORKERS:
        s, w, start, count = queue.pop(0)
        log = os.path.join(ANALYSIS, '_dephase_nb_%s_w%d.log' % (s, w))
        cmd = [PY, '-u', os.path.join(ANALYSIS, 'dephase_generate.py'),
               '--state', s, '--sahp-ainc-slow', str(STATES[s]),
               '--noise-seed-base', str(NOISE_SEED_BASE),
               '--start', str(start), '--count', str(count),
               '--duration', str(DURATION_MS),
               '--voltage', VOLTAGE,
               '--voltage-probe-n', str(VOLTAGE_PROBE_N),
               '--voltage-dt', str(VOLTAGE_DT),
               '--discard-extra-ms', str(DISCARD_EXTRA_MS)]
        running.append((s, w, subprocess.Popen(cmd, stdout=open(log, 'w'),
                                               stderr=subprocess.STDOUT), log))
        print('launched %-8s w%d (%d recordings)' % (s, w, count), flush=True)
        time.sleep(15)
    still = []
    for s, w, p, log in running:
        if p.poll() is None:
            still.append((s, w, p, log))
        else:
            done_jobs.append((s, w, p.returncode, log))
            print('  finished %-8s w%d exit=%s' % (s, w, p.returncode), flush=True)
    running = still
    if running or queue:
        have = {s: len(glob.glob(os.path.join(out_dir(s), 'recording*.npz')))
                for s in STATES_TO_RUN}
        print('[%5.1f min] %s | %d running, %d queued'
              % ((time.time()-t0)/60,
                 '  '.join('%s %d/%d' % (s, have[s], N_RECORDINGS) for s in STATES_TO_RUN),
                 len(running), len(queue)), flush=True)
        time.sleep(120)

print('\nall jobs exited after %.1f min' % ((time.time()-t0)/60))
bad = [(s, w, rc, log) for s, w, rc, log in done_jobs if rc]
for s, w, rc, log in bad:
    print('  FAILED %-8s w%d (exit %s):' % (s, w, rc))
    print('    ' + '\n    '.join(open(log).read().strip().splitlines()[-6:]))
for s in STATES_TO_RUN:
    d = out_dir(s)
    print('%-8s: %d recordings, %d rasters, %d summaries'
          % (s, len(glob.glob(os.path.join(d, 'recording*.npz'))),
             len(glob.glob(os.path.join(d, 'recording*_raster*.png'))),
             len(glob.glob(os.path.join(d, '_summary_*.json')))))
if bad:
    print('\n%d job(s) failed - check the logs above before using the dataset.' % len(bad))

## 4. Validate

Runs the four gating checks per state: population Vm has no ~4.9 s
excursion, zero bursts in the flagship's 4.60-5.34 s IC band, mean rate near
0.2789 Hz, V_rest near -83.3 mV. The second cell reports per-state summaries and
whether the residual t~0 settling event clusters by snapshot.

In [ ]:
for s in STATES_TO_RUN:
    print('=' * 72)
    print('VALIDATING state: %s  (sahp_ainc_slow = %.4f)' % (s, STATES[s]))
    print('=' * 72)
    r = subprocess.run([PY, '-u', os.path.join(ANALYSIS, 'dephase_validate.py'),
                        '--state', s], capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print('STDERR:', r.stderr[-1500:])
    print()

In [ ]:
# Per-state summary, and whether the residual t~0 settling event clusters by
# snapshot. Measured on the pilot: it does NOT (2 of 4 noise seeds burst on the
# same snapshot), so a bigger library would not remove it -- DISCARD_EXTRA_MS is
# the fix. This cell re-checks that on your own data.
sys.path.insert(0, REPO)
from neuron_simulation.analysis import detect_network_bursts
import collections

for s in STATES_TO_RUN:
    d = out_dir(s)
    fs = [p for p in sorted(glob.glob(os.path.join(d, 'recording*.npz')))
          if 'raster' not in os.path.basename(p)]
    if not fs:
        print('%-8s: no recordings' % s); continue
    rows = []
    for p in fs:
        z = np.load(p, allow_pickle=True)
        st = [np.atleast_1d(np.asarray(t, float)) for t in z['spike_times']]
        n = len(st)
        b = detect_network_bursts({j: st[j] for j in range(n)}, n,
                                  float(z['duration']),
                                  participation_threshold=0.35, burn_in_ms=0.0)
        rows.append(dict(snap=int(z['snapshot_index']) if 'snapshot_index' in z else -1,
                         n_bursts=len(b),
                         early=sum(1 for x in b if x['start_ms'] < 200.0),
                         rate=sum(len(t) for t in st) / (n * float(z['duration'])/1000.0)))
    by_snap = collections.defaultdict(lambda: [0, 0])
    for x in rows:
        by_snap[x['snap']][0] += 1
        by_snap[x['snap']][1] += x['early']
    n_early = sum(1 for x in rows if x['early'])
    print('%-8s: %d recordings | mean rate %.4f Hz (sd %.4f) | %.2f bursts/rec '
          '(0.35 gate) | %d with a t~0 event'
          % (s, len(rows), np.mean([x['rate'] for x in rows]),
             np.std([x['rate'] for x in rows]),
             np.mean([x['n_bursts'] for x in rows]), n_early))
    for k in sorted(by_snap):
        tot, early = by_snap[k]
        print('     snapshot %2d: %2d recordings, %2d with t~0 event (%.0f%%)'
              % (k, tot, early, 100.0*early/max(tot, 1)))
    if n_early and max(v[1]/max(v[0], 1) for v in by_snap.values()) > 0.5:
        print('     => concentrated in particular snapshots; raise DISCARD_EXTRA_MS')
    elif n_early:
        print('     => spread across snapshots (noise-driven, not snapshot-locked)')
    else:
        print('     => no t~0 events')
    print()

## 5. Next steps once the datasets exist

Output layout, mirroring the flagship's `normal/` + `seizure/` split:

```
notebooks/NEURON data parallel/dephased_ic/
    normal/     recordingNNN.npz  recordingNNN_raster.png
                recordingNNN_raster_shuffled.png  _summary_NNN.json
                network_dephased.npz
    seizure/    (same)
```

Each `recordingNNN.npz` carries the **full flagship field layout** — `spike_times`,
`cluster_spike_data`, `resampled_*`, `burst_windows`, `interburst_windows`,
voltage — plus branch fields `snapshot_index`, `snapshot_time_ms`, `init_mode`,
`state_name`, `sahp_ainc_slow`, `discard_transient_ms`.

None of the `single-knob-final-v1` numbers transfer. Each state needs its own
characterization and its own GLM run:

```bash
# burst windows at the project's 0.35 gate (the 0.8 gate finds ~nothing here,
# because 25 of the flagship's 26 stored 0.8-gate bursts WERE the IC event)
python analysis/burst_windows_p035.py

# GLM at the shipped operating point
python analysis/a1_typing_fix.py --n-recordings 50
```

Both of those currently point at the flagship session directory — retarget them
at `dephased_ic/<state>/` before running.

**Expected differences from the flagship:** mean rate a few percent higher (the
flagship's rate is depressed by post-IC-burst adaptation this branch no longer
has — the pilot measured +4.0% excluding the first 2 s), and every burst statistic
changed.

**A note on the seizure state.** `sahp_ainc_slow = 0.004` weakens slow adaptation,
which is what raises firing and loosens bursting. Since the warm-start library is
built by running that same network for 130 s, the seizure snapshots will carry a
*different* tonic `g_slow` than the normal ones — that is correct and expected, not
a bug. The preflight refuses to mix them.